In [14]:
import pickle
import json
from pathlib import Path
import networkx as nx
import numpy as np
import faiss


from transformers import AutoTokenizer, AutoModel
from collections import defaultdict
import math

print("Imports OK")


Imports OK


In [15]:
graph_path = Path("../schemas/schema_graph_academic.pkl")

if not graph_path.exists():
    raise FileNotFoundError("Schema graph not found. Run Phase 1.")

with open(graph_path, "rb") as f:
    G = pickle.load(f)

print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Graph loaded
Nodes: 57
Edges: 42


In [16]:
# Rebuild schema text corpus
schema_texts = []
schema_ids = []

for node, data in G.nodes(data=True):
    if data["type"] == "table":
        cols = [
            n.split(".")[1]
            for n in G.neighbors(node)
            if G.nodes[n]["type"] == "column"
        ]
        text = f"table {node} with columns " + ", ".join(cols)
        schema_texts.append(text)
        schema_ids.append(node)

    elif data["type"] == "column":
        table, col = node.split(".")
        text = f"column {col} in table {table}"
        schema_texts.append(text)
        schema_ids.append(node)

print("Schema items:", len(schema_texts))


Schema items: 57


In [18]:
# use your custom HF embedder (already defined above)
embeddings = embedder.encode(
    schema_texts,
    normalize_embeddings=True
).astype("float32")   # FAISS requires float32

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print("FAISS index ready:", faiss_index.ntotal)


FAISS index ready: 57


In [19]:
def tokenize(text):
    return text.lower().split()

doc_tokens = [tokenize(t) for t in schema_texts]

df = defaultdict(int)
for tokens in doc_tokens:
    for t in set(tokens):
        df[t] += 1

N = len(doc_tokens)

def bm25_score(query, doc_tokens, k1=1.5, b=0.75):
    q_tokens = tokenize(query)
    scores = []

    avgdl = sum(len(d) for d in doc_tokens) / N

    for tokens in doc_tokens:
        score = 0.0
        dl = len(tokens)
        for q in q_tokens:
            if q not in tokens:
                continue
            tf = tokens.count(q)
            idf = math.log((N - df[q] + 0.5) / (df[q] + 0.5) + 1)
            score += idf * ((tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl)))
        scores.append(score)

    return scores


In [20]:
def khop_tables(seed_tables, k=2):
    visited = set(seed_tables)
    frontier = set(seed_tables)

    for _ in range(k):
        next_frontier = set()
        for node in frontier:
            for n in G.neighbors(node):
                if G.nodes[n]["type"] == "table" and n not in visited:
                    visited.add(n)
                    next_frontier.add(n)
        frontier = next_frontier

    return visited


In [21]:
def find_join_paths(tables):
    paths = {}
    tables = list(tables)

    for i in range(len(tables)):
        for j in range(i + 1, len(tables)):
            t1, t2 = tables[i], tables[j]
            try:
                path = nx.shortest_path(G, t1, t2)
                paths[(t1, t2)] = path
            except nx.NetworkXNoPath:
                continue

    return paths


In [22]:
def graphrag_cartographer(question, top_k=8, khop=2):
    # --- FAISS retrieval ---
    q_emb = embedder.encode([question], normalize_embeddings=True)
    scores, idxs = faiss_index.search(q_emb, top_k)

    retrieved = [schema_ids[i] for i in idxs[0]]

    # --- BM25 rerank ---
    bm25 = bm25_score(question, doc_tokens)
    combined = []

    for i, sid in enumerate(schema_ids):
        if sid in retrieved:
            combined.append((sid, bm25[i]))

    combined.sort(key=lambda x: x[1], reverse=True)

    seed_tables = set(
        sid.split(".")[0] for sid, _ in combined[:5]
    )

    # --- Graph expansion ---
    expanded_tables = khop_tables(seed_tables, k=khop)

    # --- Join paths ---
    join_paths = find_join_paths(expanded_tables)

    # --- Ambiguity detection ---
    ambiguous = len(seed_tables) > 1 and len(join_paths) == 0

    return {
        "question": question,
        "seed_tables": list(seed_tables),
        "expanded_tables": list(expanded_tables),
        "join_paths": join_paths,
        "ambiguous": ambiguous
    }


In [23]:
q = "List students and the courses they are enrolled in"

result = graphrag_cartographer(q)

print("Question:", result["question"])
print("Seed tables:", result["seed_tables"])
print("Expanded tables:", result["expanded_tables"])
print("Ambiguous:", result["ambiguous"])

print("\nJOIN paths:")
for k, v in result["join_paths"].items():
    print(k, "->", v)


Question: List students and the courses they are enrolled in
Seed tables: ['domain_conference', 'conference', 'organization', 'author']
Expanded tables: ['domain_conference', 'conference', 'organization', 'author']
Ambiguous: True

JOIN paths:
